## Funnel Analysis

The goal is to perform funnel analysis for an e-commerce website.

Typically, websites have a clear path to conversion: for instance, you land on the home page, then you search, select a product and buy it. At each of these steps, some users will drop off and leave the site. The sequence of pages that leads to conversion is called ‘funnel’ .

Data Science can have a tremendous impact on funnel optimization.
Funnel analysis allows to understand where/when our users abandon the website. It gives crucial insights on user behavior and on ways to improve the user experience as well as it often allows to discover bugs.

In [0]:
# how to install new modules
#pip install seaborn

In [0]:
# Check available top-level paths
display(dbutils.fs.ls("dbfs:/"))

# Expected output if DBFS root is disabled:
# - Volumes/
# - Workspace/
# - databricks-datasets/
dbutils.fs.ls("dbfs:/Volumes/")

![image.png](attachment:image.png)

In [0]:
#importing libraries 

#data analysis modules
import pandas as pd 
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when

#plotting
import matplotlib.pyplot as plt 
import seaborn as sns

#interacting with operating system
import os

In [0]:
CATALOG_NAME = "workspace"
SCHEMA_NAME = "default"
VOLUME_NAME = "course_data"

user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
workspace_path = f"/Workspace/Users/{user}/bda_course/BDA2/data/"
volume_path = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}/funnel_data/"

# Copy only the specific CSV files we need to Volume so Spark can read them
files = ['user_table.csv', 'home_page_table.csv', 'search_page_table.csv', 
         'payment_page_table.csv', 'payment_confirmation_table.csv']

for file in files:
    dbutils.fs.cp(workspace_path + file, volume_path + file)

# Loading data as Spark DataFrames from Volume
user_page = spark.read.csv(volume_path+'user_table.csv', header=True, inferSchema=True)
home_page = spark.read.csv(volume_path+'home_page_table.csv', header=True, inferSchema=True)
search_page = spark.read.csv(volume_path+'search_page_table.csv', header=True, inferSchema=True)
payment_page = spark.read.csv(volume_path+'payment_page_table.csv', header=True, inferSchema=True)
confirmation_page = spark.read.csv(volume_path+'payment_confirmation_table.csv', header=True, inferSchema=True)

In [0]:
#having a brief look at the loaded dataframes. 
print("User Page:")
user_page.show(5)
print("\nHome Page:")
home_page.show(5)
print("\nSearch Page:")
search_page.show(5)
print("\nPayment Page:")
payment_page.show(5)
print("\nConfirmation Page:")
confirmation_page.show(5)

You are looking at data from an e-commerce website. The site is very simple and has just 4 pages:


- The first page is the home page. When you come to the site for the first time, you can only land on the home page as a first page.

- From the home page, the user can perform a search and land on the search page. 

- From the search page, if the user clicks on a product, she will get to the payment page, where she is asked to provide payment information in order to buy that product.

- If she does decide to buy, she ends up on the confirmation page


In [0]:
#merging the tables together into one Spark DataFrame
# First, rename the 'page' columns before joining to avoid ambiguity
home_page_renamed = home_page.withColumnRenamed('page', 'page_home')
search_page_renamed = search_page.withColumnRenamed('page', 'page_search')
payment_page_renamed = payment_page.withColumnRenamed('page', 'page_payment')
confirmation_page_renamed = confirmation_page.withColumnRenamed('page', 'page_confirmation')

# Perform left joins
df = user_page.join(home_page_renamed, on='user_id', how='left')
df = df.join(search_page_renamed, on='user_id', how='left')
df = df.join(payment_page_renamed, on='user_id', how='left')
df = df.join(confirmation_page_renamed, on='user_id', how='left')

display(df)

In [0]:
# Replace page names with 1 and fill nulls with 0
df = df.withColumn('page_home', when(col('page_home') == 'home_page', 1).otherwise(col('page_home'))) \
       .withColumn('page_search', when(col('page_search') == 'search_page', 1).otherwise(col('page_search'))) \
       .withColumn('page_payment', when(col('page_payment') == 'payment_page', 1).otherwise(col('page_payment'))) \
       .withColumn('page_confirmation', when(col('page_confirmation') == 'payment_confirmation_page', 1).otherwise(col('page_confirmation')))

# Fill nulls with 0 for all columns
df = df.fillna(0)

display(df)

In [0]:
# Calculate sums using Spark aggregation
total_list = [
    ['page_home', df.agg(F.sum('page_home')).collect()[0][0]],
    ['page_search', df.agg(F.sum('page_search')).collect()[0][0]],
    ['page_payment', df.agg(F.sum('page_payment')).collect()[0][0]],
    ['page_confirmation', df.agg(F.sum('page_confirmation')).collect()[0][0]]
]

In [0]:
total = pd.DataFrame(total_list, columns = ['page','sum'])
display(total)

In [0]:
total


In [0]:
import matplotlib.pyplot as plt

# Sample data
sizes = total['sum']
labels =total['page']

# Create a pie chart
plt.pie(sizes, labels=labels, autopct='%1.1f%%')

# Add a title
plt.title('Pie Chart')

# Display the chart
plt.show()

In [0]:
#Visualizing barplot for page visits.
# sns.barplot(x ='page', y = 'sum', data = total)

# Create a bar plot with Seaborn
ax = sns.barplot(x='page', y='sum', data=total)

# Add data labels to the bars
for p in ax.patches:
    ax.annotate(format(p.get_height(), '.0f'), (p.get_x() + p.get_width() / 2, p.get_height()), ha = 'center', va = 'center', xytext = (0, 5), textcoords = 'offset points')

# Display the plot
plt.show()

In [0]:
#The function below shows the basic statistical makeup of a specific feature
def stat(spark_df):
    '''
    INPUT: Spark DataFrame
    OUTPUT: Dataframe and plot that contains the mean of the conversion of the Input
    '''
    # Calculate means using Spark
    mean = [
        ['page_home', spark_df.agg(F.mean('page_home')).collect()[0][0]],
        ['page_search', spark_df.agg(F.mean('page_search')).collect()[0][0]],
        ['page_payment', spark_df.agg(F.mean('page_payment')).collect()[0][0]],
        ['page_confirmation', spark_df.agg(F.mean('page_confirmation')).collect()[0][0]]
    ]
    mean = pd.DataFrame(mean, columns=['page', 'mean'])
    
    print(mean)
    
    #plotting the graph of funnel analysis based on the feature
    fig, ax = plt.subplots(figsize=(8, 5))
    ax = sns.barplot(x='page', y='mean', data=mean)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.2%}', (p.get_x() + p.get_width() / 2, p.get_height()), 
                   ha='center', va='center', xytext=(0, 5), textcoords='offset points')
    ax.set_xlabel('Page', fontsize=12)
    ax.set_ylabel('Ratio of Visitors', fontsize=12)
    plt.show()

In [0]:
#Viewing descriptive statistics of 'df' DataFrame
#function allows me to reuse the code 
stat(df)

In [0]:
#Finding the unique values in each feature using Spark
print("Device values:")
df.select('device').distinct().show()
print("\nSex values:")
df.select('sex').distinct().show()

In [0]:
#Separating the original dataset into separate features using Spark filtering
df_desktop = df.filter(col('device') == 'Desktop')
df_mobile = df.filter(col('device') == 'Mobile')
df_male = df.filter(col('sex') == 'Male')
df_female = df.filter(col('sex') == 'Female')

In [0]:
#Visualizing descriptive statisitical make up of seperate features.
print(stat(df_male))
print(stat(df_female))
print(stat(df_desktop))
print(stat(df_mobile))

The above funnel analysis shows that between the homepage and searchpage, the customers churn by 50%. This means that out of all the people that make it to the homepage only half of them will go through the searchpage. 

However, it is more surprising to note that the churn between search page and payment page is greater than half. This is where most of the customers who do make it to the searchpage, ultimately choose to not go through with the payment. This can be due to a variety of reasons. The most important cause being that the search algorithm that is implemented on the website is not effective enough. This is implied from the fact that the greater magnitude of churn indicates that the customer has not found the product that he or she desires. 

There are two main features that were explored inorder to further determine the cause of churn - sex and device. While looking at the difference between the male and female features, it turns out that difference between the churn is not that much, implying that the approach taken to target these two segments of customers is effective, since we are able to convert them, enough though it is not efficient enough. 

However, when we look at the churn of customers depending on their device, it is a different story. While one may assume that the customer churn will be lower on desktop than on mobile simply because, usage of desktop implies the customer is more serious about the purchase rather than the impulsive customer on the mobile. The data above does not support this claim, infact that customer on the mobile is more likely to purchase the product than on desktop. This implies that the payment interface on the mobile is more customer friendly than that of the desktop. 

The above analysis is not the end but rather the starting point of improving the conversion rate of the ecommerce website.